In [1]:
#Import libraries
import json
import re
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_DIR = Path.cwd().parent

RULES_PATH = PROJECT_DIR / "knowledge_base" / "conduct_rules.json"

print("Project directory:", PROJECT_DIR)
print("Rules file exists:", RULES_PATH.exists())

Project directory: c:\Users\user\Desktop\ConductIQ
Rules file exists: True


In [3]:
with open(RULES_PATH, "r", encoding="utf-8") as file:
    knowledge_base = json.load(file)

rules = knowledge_base["rules"]

print("Total rules:", len(rules))

Total rules: 29


In [4]:
# Validate the knowledge base
required_fields = [
    "rule_id",
    "rule_name",
    "category",
    "section_number",
    "description",
    "possible_measures",
    "conditions",
    "exceptions",
    "source_quote",
    "source_page",
    "retrieval_keywords"
]

for rule in rules:
    missing = [
        field for field in required_fields
        if field not in rule
    ]

    if missing:
        print(rule["rule_id"], "Missing:", missing)

rule_ids = [rule["rule_id"] for rule in rules]

print("Unique IDs:", len(set(rule_ids)))
print("Duplicate IDs:", len(rule_ids) - len(set(rule_ids)))

Unique IDs: 29
Duplicate IDs: 0


In [5]:
# Create the retrieval engine 
class ConductRuleEngine:

    STOPWORDS = {
        "what", "when", "where", "why", "how",
        "is", "are", "was", "were", "a", "an", "the",
        "if", "of", "in", "on", "at", "to", "for",
        "and", "or", "but", "can", "could", "would",
        "should", "do", "does", "did", "my", "me",
        "i", "student", "happens", "happen", "about"
    }

    def __init__(self, rules):
        self.rules = rules

    @staticmethod
    def normalize(text):
        return re.sub(
            r"[^a-z0-9\s]",
            " ",
            str(text).lower()
        ).strip()

    def tokenize(self, text):
        words = self.normalize(text).split()

        return {
            word for word in words
            if word not in self.STOPWORDS and len(word) > 1
        }

    def search_rules(self, query, top_k=5):

        query_normalized = self.normalize(query)
        query_terms = self.tokenize(query)

        if not query_terms:
            return []

        results = []

        for rule in self.rules:

            name_terms = self.tokenize(
                rule.get("rule_name", "")
            )

            category_terms = self.tokenize(
                rule.get("category", "")
            )

            description_terms = self.tokenize(
                rule.get("description", "")
            )

            quote_terms = self.tokenize(
                rule.get("source_quote", "")
            )

            keyword_terms = set()

            for keyword in rule.get("retrieval_keywords", []):
                keyword_terms.update(self.tokenize(keyword))

            score = 0
            matched_terms = []

            for term in query_terms:

                if term in name_terms:
                    score += 3
                    matched_terms.append(term)

                elif term in keyword_terms:
                    score += 3
                    matched_terms.append(term)

                elif term in category_terms:
                    score += 2
                    matched_terms.append(term)

                elif term in description_terms:
                    score += 1
                    matched_terms.append(term)

                elif term in quote_terms:
                    score += 0.5
                    matched_terms.append(term)

            # Additional bonus for exact phrase matches.
            phrase_bonus = 0

            for keyword in rule.get("retrieval_keywords", []):
                normalized_keyword = self.normalize(keyword)

                if normalized_keyword in query_normalized:
                    phrase_bonus = max(phrase_bonus, 3)

            normalized_name = self.normalize(
                rule.get("rule_name", "")
            )

            if normalized_name in query_normalized:
                phrase_bonus += 3

            final_score = score / len(query_terms) + phrase_bonus

            if final_score > 0:
                result = rule.copy()

                result["retrieval_score"] = round(
                    final_score, 4
                )

                result["matched_terms"] = sorted(
                    set(matched_terms)
                )

                results.append(result)

        results.sort(
            key=lambda x: x["retrieval_score"],
            reverse=True
        )

        return results[:max(0, top_k)]

    def get_rule_by_id(self, rule_id):

        for rule in self.rules:
            if rule["rule_id"] == rule_id:
                return rule

        return None

In [6]:
# Initialize the engine
rule_engine = ConductRuleEngine(rules)

print("ConductIQ Rule Engine initialized successfully.")

ConductIQ Rule Engine initialized successfully.


In [7]:
# test retrival 
test_questions = [
    "What happens if I plagiarize an assignment?",
    "What if someone forges a certificate?",
    "Can students bully other students?",
    "What happens if someone steals a laptop?",
    "What if someone damages college property?",
    "Can students misuse social media?",
    "What happens if someone is caught drinking alcohol?",
    "What if someone secretly records another student?"
]

for question in test_questions:

    print("\n" + "=" * 70)
    print("QUESTION:", question)

    results = rule_engine.search_rules(question, top_k=3)

    for result in results:

        print("\nRule:", result["rule_name"])
        print("Rule ID:", result["rule_id"])
        print("Section:", result["section_number"])
        print("Page:", result["source_page"])
        print("Score:", result["retrieval_score"])
        print("Matched terms:", result["matched_terms"])


QUESTION: What happens if I plagiarize an assignment?

Rule: Academic Dishonesty
Rule ID: CONDUCT_001
Section: 3.1.1
Page: 9
Score: 4.5
Matched terms: ['plagiarize']

QUESTION: What if someone forges a certificate?

Rule: Forgery
Rule ID: CONDUCT_003
Section: 3.1.3
Page: 9
Score: 4.0
Matched terms: ['certificate']

Rule: False Charges
Rule ID: CONDUCT_004
Section: 3.1.4
Page: 10
Score: 1.0
Matched terms: ['someone']

Rule: Bullying/ragging
Rule ID: CONDUCT_005
Section: 3.2.1
Page: 10
Score: 1.0
Matched terms: ['someone']

QUESTION: Can students bully other students?

Rule: Bullying/ragging
Rule ID: CONDUCT_005
Section: 3.2.1
Page: 10
Score: 4.0
Matched terms: ['bully']

Rule: Disruptive Conduct
Rule ID: CONDUCT_024
Section: 3.5.4
Page: 15
Score: 0.6667
Matched terms: ['other', 'students']

Rule: Unauthorised Collection of Funds
Rule ID: CONDUCT_020
Section: 3.4.4
Page: 14
Score: 0.5
Matched terms: ['other', 'students']

QUESTION: What happens if someone steals a laptop?

Rule: Stealin

In [8]:
# inspect a specific rule 
rule = rule_engine.get_rule_by_id("CONDUCT_001")

if rule:
    print("Rule:", rule["rule_name"])
    print("Section:", rule["section_number"])
    print("Description:", rule["description"])
    print("Official quotation:", rule["source_quote"])
    print("Page:", rule["source_page"])

Rule: Academic Dishonesty
Section: 3.1.1
Description: Academic dishonesty is defined and dealt with under the Wheel of Academic Law and other applicable University policies and regulations.
Official quotation: Academic dishonesty shall be defined and dealt with in accordance with the provisions in the Wheel of Academic Law and other applicable University policies and regulations.
Page: 9
